<a href="https://colab.research.google.com/github/muneer-ahmad10/Natural-Language-processing/blob/main/Day_06_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



*   RNN has a limitation of vanishing gradient which is solved using LSTM
*   RNN =Short Memory
*   LSTM=Smart memory
*  LSTM uses gates(input output forget)




In [52]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence

In [53]:
# 🔹 2. Data
texts = [
    "i love this movie",
    "this is amazing",
    "i hate this",
    "this is terrible"
]
labels = [1, 1, 0, 0]  # 1 = positive, 0 = negative

In [54]:
all_texts=' '.join(texts).split()

In [55]:
all_texts

['i',
 'love',
 'this',
 'movie',
 'this',
 'is',
 'amazing',
 'i',
 'hate',
 'this',
 'this',
 'is',
 'terrible']

In [56]:
labels=torch.tensor(labels,dtype=torch.float32).unsqueeze(1)

In [57]:
lebels

tensor([[1.],
        [1.],
        [0.],
        [0.]])

In [58]:
vocab={word: i+1 for i , word in enumerate(set(all_texts))}
vocab

{'this': 1,
 'amazing': 2,
 'terrible': 3,
 'movie': 4,
 'i': 5,
 'is': 6,
 'love': 7,
 'hate': 8}

In [59]:
def encode(sentence):
  return[vocab[word] for word in sentence.split()]

In [60]:
encoded_text=[torch.tensor(encode(t)) for t in texts]

In [61]:
encoded_text

[tensor([5, 7, 1, 4]), tensor([1, 6, 2]), tensor([5, 8, 1]), tensor([1, 6, 3])]

In [62]:
padded=pad_sequence(encoded_text,batch_first=True)

In [63]:
padded

tensor([[5, 7, 1, 4],
        [1, 6, 2, 0],
        [5, 8, 1, 0],
        [1, 6, 3, 0]])

In [64]:
class LSTMmodel(nn.Module):
  def __init__(self,vocab_size,embed_dim,hidden_dim):
    super().__init__()

    self.embed=nn.Embedding(vocab_size,embed_dim)
    self.lstm=nn.LSTM(embed_dim,hidden_dim,batch_first=True)
    self.fc=nn.Linear(hidden_dim,1)
    self.sigmoid=nn.Sigmoid()

  def forward(self,x):
    x=self.embed(x)
    output,(hidden,cell)=self.lstm(x)
    out=self.fc(hidden[-1])
    return self.sigmoid(out)

In [65]:

model=LSTMmodel(vocab_size=len(vocab)+1,embed_dim=8,hidden_dim=16)

In [66]:
criterion=nn.BCELoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.01)

In [70]:
for epoch in range(100):
  optimizer.zero_grad()

  output=model(padded)

  loss=criterion(output,labels)

  loss.backward()
  optimizer.step()

  if epoch % 20 ==0:
    print(f'Epochs : {epoch} =>  loss : {loss.item()}')


Epochs : 0 =>  loss : 0.0010062051005661488
Epochs : 20 =>  loss : 0.000834768230561167
Epochs : 40 =>  loss : 0.0007066676043905318
Epochs : 60 =>  loss : 0.0006073921686038375
Epochs : 80 =>  loss : 0.0005286685191094875


In [71]:
print(f"Number of texts: {len(all_texts)}")
print(f"Number of labels: {len(labels)}")

Number of texts: 13
Number of labels: 4


In [72]:
def predict(sentence):
  encoded=torch.tensor(encode(sentence)).unsqueeze(0)
  output=model(encoded)
  return "Positive" if output.item() > 0.5 else "Negative"

print(predict('i love this'))
print(predict('i hate this'))

Positive
Negative
